# Time some basic operations in movie with varying sizes

Notebook to automatically do time some steps of EpiCure:
* **Load the movie and segmentation**
* **removing the border cells** 
* **performs the tracking** (with the default parameters)
* **perform some basic editions, repeated several times** 

*This notebook is part of EpiCure release, see https://github.pasteur.fr/Image-Analysis-Hub/epicure for more informations*

In [ ]:
#### Parent directory to process. It will go through all the sub-folders
main_path = "../../../data/timing/"
imgname = main_path+"Ecad"
skelname = main_path+"Ecad_skeleton"
#subnames = ["-subcrop", "-crop-crop", "-crop", "", "-subset", "-subset-subset"]
subnames = ["", "-subset", "-subset-subset"]

In [ ]:
import epicure.epicuring as epicure
import epicure.Utils as ut
import os
import time
from vispy import keys
import sys

original_stdout = sys.stdout

def doOneMovie( imgname, skelname, subname ):
    """ Process one movie: clear (or not) the border cells and performs tracking """
    print( "EpiCuring movie "+imgname+subname+".tif" )
    
    start_time = time.time()
    sys.stdout = open( os.devnull, 'w' )
    epic = epicure.EpiCure()
    epic.verbose = 0         ## Choose the level of printing infos during process (0: minimal to 3: debug informations)
    ## Loading movie
    epic.load_movie( imgname+subname+".tif" )
    sys.stdout = original_stdout
    ctime = time.time()
    print( f"Time to load movie: {ctime-start_time}" )
    ## Loading segmentation and initialize all
    start_time = ctime
    sys.stdout = open( os.devnull, 'w' )
    epic.go_epicure( "epics", skelname+subname+".tif" )  ## start EpiCure, load the segmentation, prepare everything
    sys.stdout = original_stdout
    ctime = time.time()
    print( f"Time to load segmentation and initialize: {ctime-start_time}" )
    print( f"Nb of untracked cells: {epic.nlabels()}" )
    ## Remove border cells
    start_time = ctime 
    sys.stdout = open( os.devnull, 'w' )
    epic.editing.border_size.setText("1") ## Choose the border size parameter to remove the cells that are within the given distance of the image border
    epic.editing.remove_border()  ## EpiCure option to remove all cells that are touching the image border
    sys.stdout = original_stdout
    ctime = time.time()
    print( f"Time to remove cells on border: {ctime-start_time}" )
    ## Track with LapTrack centroids, no features
    start_time = ctime
    sys.stdout = open( os.devnull, 'w' )
    epic.tracking.do_tracking()       ## Performs tracking with the default parameters. If you have saved preferences, it will use it.
    sys.stdout = original_stdout
    ctime = time.time()
    print( f"Time to track: {ctime-start_time}" )
    ## Do some basic editings to measure reactivness
    start_time = ctime    
    view = epic.viewer.window.qt_viewer
    for i in range(50):
        view.canvas.events.key_press(key=keys.Key("b"), modifiers=[])
        view.canvas.events.key_press(key=keys.Key("c"), modifiers=[])
        view.canvas.events.key_press(key=keys.Key("v"), modifiers=[])
    ctime = time.time()
    print( f"Time to change view: {ctime-start_time}" )
    ## Get neighbor labels
    start_time = ctime
    segedit = epic.editing
    label = epic.seg[0,200,200]
    graph = ut.get_neighbor_graph( epic.seg[0], distance=3 )
    neighbors = ut.get_neighbors( label, graph )
    neigh = neighbors[0]
    ## Hoping labels are neighbors
    start_time = ctime
    segedit = epic.editing
    sys.stdout = open( os.devnull, 'w' )
    for i in range(30):
        segedit.merge_labels( 0, label, neigh )
        view.canvas.events.key_press(key=keys.Key("Z"), modifiers=["Control"])
    sys.stdout = original_stdout
    ctime = time.time()
    print( f"Time to merge and cancel: {ctime-start_time}" )
    
    start_time = ctime
    sys.stdout = open( os.devnull, 'w' )
    epic.save_epicures()       ## save the results in the ouput "epics" folder(s)
    sys.stdout = original_stdout
    ctime = time.time()
    print( f"Time to save the segmentation/epic infos: {ctime-start_time}" )
    del epic

## be sure to start from scratch
if os.path.exists( os.path.join(main_path+"epics/") ):
    import shutil
    shutil.rmtree(os.path.join(main_path+"epics/"))

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
for subname in subnames:
    doOneMovie( imgname, skelname, subname )